In [1]:
import pandas as pd
import google.generativeai as genai
import os

from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv(
    "GEMINI_API_KEY"
)

genai.configure(
    api_key=GEMINI_API_KEY
)

model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

print("Gemini connected.")

Gemini connected.


C:\Users\Samad\AppData\Local\Temp\ipykernel_23620\2468347665.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
drivers = pd.read_csv(
    "../data/drivers.csv"
)

constructors = pd.read_csv(
    "../data/constructors.csv"
)

races = pd.read_csv(
    "../data/races.csv"
)

results = pd.read_csv(
    "../data/results.csv"
)

print("Data loaded.")

Data loaded.


In [14]:
def get_driver_info(name):

    mask = (
        drivers["forename"]
        .fillna("")
        .str.cat(
            drivers["surname"].fillna(""),
            sep=" "
        )
        .str.contains(
            name,
            case=False,
            na=False
        )
    )

    return drivers[mask].head(5)

In [15]:
def get_constructor_info(name):

    return constructors[
        constructors["name"]
        .str.contains(
            name,
            case=False,
            na=False
        )
    ].head(5)

In [16]:
def get_race_info(name):

    return races[
        races["name"]
        .str.contains(
            name,
            case=False,
            na=False
        )
    ].head(10)

In [17]:
def analyze_question(question):

    prompt = f"""
You are an F1 routing assistant.

Extract all relevant entities.

Return only:

DRIVER=<name or NONE>
CONSTRUCTOR=<name or NONE>
RACE=<name or NONE>

Question:
{question}
"""

    response = model.generate_content(
        prompt
    )

    return response.text

In [18]:
response = analyze_question(
    "Tell me about Lewis Hamilton and Mercedes"
)

print(response)

DRIVER=Lewis Hamilton
CONSTRUCTOR=Mercedes
RACE=NONE


In [19]:
def parse_entities(text):

    entities = {}

    for line in text.split("\n"):

        if "=" in line:

            key, value = line.split(
                "=",
                1
            )

            entities[
                key.strip()
            ] = value.strip()

    return entities

In [20]:
entities = parse_entities(
    response
)

print(entities)

{'DRIVER': 'Lewis Hamilton', 'CONSTRUCTOR': 'Mercedes', 'RACE': 'NONE'}


In [21]:
driver_data = ""

if entities["DRIVER"] != "NONE":

    driver_data = get_driver_info(
        entities["DRIVER"]
    )

driver_data

,driverId,driverRef,number,code,forename,surname,dob,nationality,url
0,1,hamilton,44,HAM,Lewis,Hamilton,1985-01-07,British,http://en.wikipedia.org/wiki/Lewis_Hamilton


In [22]:
constructor_data = ""

if entities["CONSTRUCTOR"] != "NONE":

    constructor_data = get_constructor_info(
        entities["CONSTRUCTOR"]
    )

constructor_data

,constructorId,constructorRef,name,nationality,url
129,131,mercedes,Mercedes,German,http://en.wikipedia.org/wiki/Mercedes-Benz_in_...


In [23]:
prompt = f"""
Question:

Tell me about Lewis Hamilton and Mercedes

Driver Data:

{driver_data}

Constructor Data:

{constructor_data}

Provide a combined answer.
"""

In [24]:
answer = model.generate_content(
    prompt
)

print(answer.text)

Lewis Hamilton, the British driver born on 1985-01-07 and known for his iconic number 44, and the German constructor Mercedes have forged one of the most dominant and successful partnerships in Formula 1 history.

Hamilton made the momentous switch to Mercedes for the 2013 season. This move ushered in an unparalleled era of dominance, particularly with the introduction of the turbo-hybrid regulations in 2014.

Together, Hamilton and Mercedes have achieved extraordinary feats:

*   **Drivers' Championships:** Lewis Hamilton has secured 6 of his record-equalling 7 FIA Formula One World Championships with Mercedes (2014, 2015, 2017, 2018, 2019, 2020).
*   **Constructors' Championships:** Mercedes itself has won an unprecedented 8 consecutive FIA Formula One Constructors' Championships (2014-2021) with Hamilton as their lead driver.
*   **Records:** During their time together, Hamilton surpassed Michael Schumacher's record for most career F1 wins and pole positions, cementing his place as 

## Conclusion

In this notebook, a multi-tool AI agent was developed to answer Formula 1 questions using information from multiple datasets simultaneously.

The agent first identifies relevant entities from a user query, retrieves information from different tools, and then combines the results into a single AI-generated response.

Key achievements:
- Extracted multiple entities from user questions.
- Retrieved data from driver and constructor datasets.
- Combined outputs from multiple tools.
- Generated unified responses using Gemini.
- Demonstrated the foundation of agentic AI workflows.

This notebook extends the chatbot from single-tool reasoning to multi-tool reasoning, making responses more informative and context-aware.